In [1]:
from numpy.random import default_rng
import scanpy as sc
# import squidpy as sq
from anndata import AnnData
import scipy
# sc.logging.print_header()
from sklearn.metrics.cluster import adjusted_rand_score
import numpy as np
import pandas as pd
import seaborn as sns
import os
import torch
import pandas as pd
import scanpy as sc
from sklearn import metrics
import multiprocessing as mp
from sklearn.metrics.cluster import normalized_mutual_info_score

In [ ]:
%load_ext autoreload
%autoreload 2

In [12]:
# Should we add 5 or 6 modalities? Noisy modalities

In [3]:
# Data source: https://github.com/liranmao/Spatial_multi_omics
# https://figshare.com/articles/dataset/Datasets_for_spatial-mux-seq/30258697

rna=sc.read_h5ad('/data/hoan/spatial_transcriptomics/data/spatial-mux-seq/RNA.h5ad')
atac=sc.read_h5ad('/data/hoan/spatial_transcriptomics/data/spatial-mux-seq/ATAC.h5ad')
h3k4=sc.read_h5ad('/data/hoan/spatial_transcriptomics/data/spatial-mux-seq/H3K4me3.h5ad')
h3k27=sc.read_h5ad('/data/hoan/spatial_transcriptomics/data/spatial-mux-seq/H3K27me3.h5ad')

/home/vanhoan310/miniconda3/envs/gnn/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [4]:
import numpy as np
import scanpy as sc
import anndata as ad
import pandas as pd

N_PCS = 100

def run_pca(adata, n_pcs=50):
    """
    Normalize + log1p + PCA.
    """
    x = adata.copy()

    # Preserve raw counts if desired
    x.layers["counts"] = x.X.copy()

    # Library-size normalization
    sc.pp.normalize_total(x, target_sum=1e4)

    # Log transformation
    sc.pp.log1p(x)

    # PCA
    sc.pp.pca(
        x,
        n_comps=n_pcs,
        svd_solver="arpack"
    )

    return x


rna_pca = run_pca(rna, N_PCS)
atac_pca = run_pca(atac, N_PCS)
h3k4_pca = run_pca(h3k4, N_PCS)
h3k27_pca = run_pca(h3k27, N_PCS)


# ============================================================
# 4. Extract PCA representations
# ============================================================

Z_rna = rna_pca.obsm["X_pca"]
Z_atac = atac_pca.obsm["X_pca"]
Z_h3k4 = h3k4_pca.obsm["X_pca"]
Z_h3k27 = h3k27_pca.obsm["X_pca"]

print("RNA PCA:", Z_rna.shape)
print("ATAC PCA:", Z_atac.shape)
print("H3K4me3 PCA:", Z_h3k4.shape)
print("H3K27me3 PCA:", Z_h3k27.shape)

/home/vanhoan310/miniconda3/envs/gnn/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


RNA PCA: (2133, 100)
ATAC PCA: (2133, 100)
H3K4me3 PCA: (2133, 100)
H3K27me3 PCA: (2133, 100)


In [5]:
adata_multi = ad.AnnData(
    X=Z_rna.copy(),
    obs=rna.obs.copy()
)

adata_multi.obsm["X_rna"] = Z_rna
adata_multi.obsm["X_atac"] = Z_atac
adata_multi.obsm["X_h3k4me3"] = Z_h3k4
adata_multi.obsm["X_h3k27me3"] = Z_h3k27

# Spatial coordinates
adata_multi.obsm["spatial_local"] = rna.obsm["spatial_local"].copy()
adata_multi.obsm["spatial_pxl"] = rna.obsm["spatial_pxl"].copy()


# ============================================================
# 6. Store PCA information
# ============================================================

adata_multi.uns["modalities"] = [
    "RNA",
    "ATAC",
    "H3K4me3",
    "H3K27me3"
]

adata_multi.uns["n_pcs"] = N_PCS


print(adata_multi)
print(adata_multi.obsm.keys())

AnnData object with n_obs × n_vars = 2133 × 100
    obs: 'in_tissue'
    uns: 'modalities', 'n_pcs'
    obsm: 'X_rna', 'X_atac', 'X_h3k4me3', 'X_h3k27me3', 'spatial_local', 'spatial_pxl'
KeysView(AxisArrays with keys: X_rna, X_atac, X_h3k4me3, X_h3k27me3, spatial_local, spatial_pxl)


In [6]:
adata_multi.obsm['spatial'] = adata_multi.obsm['spatial_pxl']
adata_multi.obsm['spatial']

array([[1768.,   11.],
       [1768.,  407.],
       [1768.,  451.],
       ...,
       [1812.,  275.],
       [1812.,  319.],
       [1812.,  363.]])

In [7]:
adata_multi

AnnData object with n_obs × n_vars = 2133 × 100
    obs: 'in_tissue'
    uns: 'modalities', 'n_pcs'
    obsm: 'X_rna', 'X_atac', 'X_h3k4me3', 'X_h3k27me3', 'spatial_local', 'spatial_pxl', 'spatial'

## Data analysis

In [8]:
from sklearn.decomposition import PCA
# from GraphST.utils import refine_label
from sklearn.preprocessing import StandardScaler
# from GraphST.utils import mclust_R
import numpy as np
from numpy import dot, array
from sklearn.cross_decomposition import CCA
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import kneighbors_graph
import time

In [9]:
from sklearn.mixture import BayesianGaussianMixture
from mvlearn.datasets import load_UCImultifeature
from mvlearn.embed import GCCA
# gcca = GCCA(n_components = 20)
# Xs_latents = gcca.fit_transform(Xs)

from mvlearn.plotting import crossviews_plot
from mvlearn.embed import DCCA
from mvlearn.embed import MCCA
from mvlearn.embed import KMCCA

In [10]:
n_clusters = 10

In [11]:
for n_modalities in range(2, 5):
    print('Number of modalities: ', n_modalities)
    start_time = time.perf_counter()
    
    ### GraphConv 
    n_neighbors = 4
    connectivity = kneighbors_graph(adata_multi.obsm['spatial'], n_neighbors=n_neighbors, include_self=False)
    # make connectivity symmetric
    connectivity = 0.5 * (connectivity + connectivity.T)
    if n_modalities==2:
        RNA_spatial = connectivity.dot(adata_multi.obsm['X_rna'])
        atac_spatial = connectivity.dot(adata_multi.obsm['X_atac'])
        Xs = [RNA_spatial-RNA_spatial.min(), atac_spatial-atac_spatial.min()] # multiview data
    elif n_modalities==3:
        RNA_spatial = connectivity.dot(adata_multi.obsm['X_rna'])
        atac_spatial = connectivity.dot(adata_multi.obsm['X_atac'])
        h3k4me3_spatial = connectivity.dot(adata_multi.obsm['X_h3k4me3'])
        Xs = [RNA_spatial-RNA_spatial.min(), atac_spatial-atac_spatial.min(),h3k4me3_spatial-h3k4me3_spatial.min()] # multiview data
    elif n_modalities==4:
        RNA_spatial = connectivity.dot(adata_multi.obsm['X_rna'])
        atac_spatial = connectivity.dot(adata_multi.obsm['X_atac'])
        h3k4me3_spatial = connectivity.dot(adata_multi.obsm['X_h3k4me3'])
        h3k27me3_spatial = connectivity.dot(adata_multi.obsm['X_h3k27me3'])
        Xs = [RNA_spatial-RNA_spatial.min(), atac_spatial-atac_spatial.min(),h3k4me3_spatial-h3k4me3_spatial.min(), 
                   h3k27me3_spatial-h3k27me3_spatial.min()] # multiview data
    else:
        print('Exceed number of modalities')
    # Xs = [data_a_train, data_b_train] # multiview data
    # mcca = MCCA(n_components = 20, regs = 1)
    mcca = KMCCA(n_components = 20, kernel = 'poly', regs = 0)
    mcca.fit(Xs)
    # MCCA()
    Xs_latents = mcca.transform(Xs)
    adata_multi.obsm['emb_pca'] = np.concatenate(
    [Xs_latents[i, :, :] for i in range(n_modalities)],
    axis=1)
    print(adata_multi.obsm['emb_pca'].shape)
    gmm = BayesianGaussianMixture(n_components=n_clusters, covariance_type='full', random_state=42, init_params = 'random_from_data', n_init = 5, max_iter = 500)
    # gmm.means_init = np.array([embedding[initial_labels == i].mean(axis=0) for i in range(n_clusters)])
    # Step 4: Fit GMM
    gmm.fit(adata_multi.obsm['emb_pca'])
    cluster_labels = gmm.predict(adata_multi.obsm['emb_pca'])

    elapsed = time.perf_counter() - start_time
    print(f"Runtime for {n_modalities} modalities: {elapsed:.3f} seconds")

Number of modalities:  2
(2133, 40)
Runtime for 2 modalities: 48.628 seconds
Number of modalities:  3
(2133, 60)
Runtime for 3 modalities: 50.013 seconds
Number of modalities:  4
(2133, 80)
Runtime for 4 modalities: 58.793 seconds
